# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa" using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a DatasetMetadata object

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
List the record sets available in the dataset, along with their `@id`, `name`, and fields.

The Croissant schema organizes data using Record Sets (`@type: RecordSet`), each of which have fields and columns defined by their `@id` values. These IDs are needed to extract data using `mlcroissant`.

In [ ]:
print("Record Sets in this dataset:\n")

for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}\n  Name: {record_set.name}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
    print()

## 3. Data Extraction
Extract data from the main record set containing survey results. All references use the `@id` fields as per the Croissant specification.

We'll load all the available record sets into separate pandas DataFrames using their `@id`.

In [ ]:
# Identify all record set @id's
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Available record set @id's:")
print(record_set_ids)

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set '{record_set_id}' with {len(df)} records, columns:")
        print(df.columns.tolist())
    except Exception as e:
        print(f"Could not load record set '{record_set_id}': {e}")
    print()
# For further analysis we'll use the main survey records set. Typically, it's named 'fair2_survey_main' or similar. If only one record set exists, use that.
if len(record_set_ids) == 1:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = record_set_ids[0]  # Adjust if you know the correct main record set

print(f"Main record set under analysis: {main_record_set_id}")

main_df = dataframes[main_record_set_id]
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Below are examples of common data cleaning and analysis steps. **All field and column references use their Croissant `@id` values** (e.g., `field:age`, `field:GAD7_score`).

We'll demonstrate filtering by a numeric field (e.g., PHQ-9 score: `field:phq9_total`), normalization, and grouping by a demographic field (e.g., Gender: `field:gender`).

Replace the field IDs below with those displayed in section 2 for your dataset.

In [ ]:
# Example field @id's (replace with actual @id's printed above):
numeric_field_id = None
group_field_id = None

# Try to infer possible field names for common psychological scales
for col in main_df.columns:
    if 'phq' in col.lower():
        numeric_field_id = col
    if ('gender' in col.lower()) or ('sex' in col.lower()):
        group_field_id = col

if numeric_field_id is None:
    numeric_field_id = main_df.select_dtypes('number').columns[0]  # fallback
    print(f"Auto-selected numeric field: {numeric_field_id}")
else:
    print(f"Selected numeric field: {numeric_field_id}")

print(f"Selected group field: {group_field_id}")

# Filter records with a threshold (e.g. only high PHQ-9 scores)
threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records in '{main_record_set_id}' with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalizing the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std() + 1e-8)
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by demographic field (if available)
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field (e.g., PHQ-9/GAD-7 scores) or compare means across demographic groups. Adjust IDs for your dataset as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the main numeric variable
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=20, kde=True, color='skyblue')
plt.title(f"Distribution of '{numeric_field_id}' scores")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group (if available)
if group_field_id is not None and group_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
    plt.show()

## 6. Conclusion

- We demonstrated how to load and inspect a Croissant-schema-based survey dataset using `mlcroissant`.
- The dataset contains valuable mental health survey results (e.g., GAD-7, PHQ-9) from Kilifi County, Kenya.
- Basic data wrangling and EDA steps (filtering, normalization, grouping, and visualization) were performed, referencing all schema elements by their Croissant `@id` fields for reproducibility and clarity.

Refer to the Croissant metadata for more advanced use cases or specific variable details. For production/data science workflows, always validate field names and dataset completeness via the schema programmatically.